In [2]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

Python: 3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)]
PyTorch: 2.11.0+cu130
CUDA available: True
CUDA version: 13.0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.0 GB


In [3]:
import unsloth

print("Unsloth:", unsloth.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\Swapn\miniconda3\envs\unsloth\Lib\site-packages\unsloth\__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: 2026.8.4


In [4]:
import transformers
import trl
import peft

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

Transformers: 5.5.0
TRL: 0.24.0
PEFT: 0.20.0


In [5]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved:  {reserved:.2f} GB")

Allocated: 0.01 GB
Reserved:  0.02 GB


In [6]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_SEQ_LENGTH = 512

LOAD_IN_4BIT = True

SEED = 3407

print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("4-bit:", LOAD_IN_4BIT)
print("Seed:", SEED)

Model: Qwen/Qwen2.5-0.5B-Instruct
Max sequence length: 512
4-bit: True
Seed: 3407


In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

==((====))==  Unsloth 2026.8.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4050 Laptop GPU. Num GPUs = 1. Max memory: 5.997 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [8]:
print("Model loaded successfully.")
print("Model type:", type(model))
print("Tokenizer:", type(tokenizer))

Model loaded successfully.
Model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Tokenizer: <class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>


In [9]:
print("Chat template available:", tokenizer.chat_template is not None)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

Chat template available: True
EOS token: <|im_end|>
EOS token ID: 151645
PAD token: <|vision_pad|>
PAD token ID: 151654


In [10]:
model = FastLanguageModel.get_peft_model(
    model,

    r=16,
    lora_alpha=32,
    lora_dropout=0.0,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",

    use_gradient_checkpointing="unsloth",
    random_state=SEED,

    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.8.4 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [11]:
print(model.peft_config)

{'default': LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'Qwen2ForCausalLM', 'parent_library': 'transformers.models.qwen2.modeling_qwen2', 'unsloth_fixed': True}, peft_version='0.20.0', base_model_name_or_path='unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit', revision=None, inference_mode=False, r=16, target_modules={'gate_proj', 'q_proj', 'up_proj', 'down_proj', 'v_proj', 'k_proj', 'o_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, 

In [12]:
from datasets import load_dataset

In [13]:
DATA_DIR = "data/sft"

dataset = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/val.jsonl",
    }
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'messages', 'text'],
        num_rows: 48616
    })
    validation: Dataset({
        features: ['id', 'messages', 'text'],
        num_rows: 1169
    })
})


In [14]:
for split in ["train", "validation"]:
    empty = sum(
        not x["text"].strip()
        for x in dataset[split]
    )

    print(f"{split}: {empty} empty examples")

    assert empty == 0

train: 0 empty examples
validation: 0 empty examples


In [15]:
print(dataset["train"].column_names)

['id', 'messages', 'text']


In [16]:
from trl import SFTTrainer, SFTConfig

In [17]:
training_args = SFTConfig(
    output_dir="outputs/qwen25_05b_qlora",

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    # Optimizer
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,

    # Precision
    fp16=False,
    bf16=True,
    warmup_steps=0.05,

    # Logging / evaluation
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,

    # Checkpoints
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    # Reproducibility
    seed=SEED,

    # Dataset
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,

    # Performance
    optim="adamw_8bit",
    gradient_checkpointing=True,

    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [18]:
import inspect

print(inspect.signature(SFTConfig))

(output_dir=None, per_device_train_batch_size=4, num_train_epochs=3.0, max_steps=-1, learning_rate=5e-05, lr_scheduler_type='linear', lr_scheduler_kwargs=None, warmup_steps=0.1, optim='adamw_8bit', optim_args=None, weight_decay=0.001, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=2, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=False, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=250, auto_find_batch_size=False, logging_strategy='steps', logging_steps=1, logging_first_step=False, log_on_each_node=True, logging_nan_inf_filter=False, include_num_input_tokens_seen=False, log_level='passive', log_level_replica='war

In [19]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,   # renamed from tokenizer=
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/48616 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/1169 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [20]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Map:   0%|          | 0/48616 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48616 [00:00<?, ? examples/s]

Unsloth: Removed 22254 out of 48616 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Map:   0%|          | 0/1169 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1169 [00:00<?, ? examples/s]

Unsloth: Removed 507 out of 1169 samples from eval_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


In [21]:
sample = dataset["train"][0]

print(sample.keys())
print(sample["text"][:3000])

dict_keys(['id', 'messages', 'text'])
<|im_start|>system
You are a function-calling assistant.

Your task is to select the appropriate tool and provide the arguments required
to answer the user's request.

Return ONLY valid JSON.

The output must be a JSON array of tool calls in this format:

[
  {
    "name": "function_name",
    "arguments": {
      "argument_name": "value"
    }
  }
]

If none of the available tools can answer the request, return an empty array: []

Do not explain your answer.
Do not output markdown.
Do not invent tools.
<|im_end|>
<|im_start|>user
Available tools:

[
  {
    "name": "live_giveaways_by_type",
    "description": "Retrieve live giveaways from the GamerPower API based on the specified type.",
    "parameters": {
      "type": {
        "description": "The type of giveaways to retrieve (e.g., game, loot, beta).",
        "type": "str",
        "default": "game"
      }
    }
  }
]

User request:

Where can I find live giveaways for beta access and games

In [22]:
encoded = tokenizer(
    sample["text"],
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
    return_tensors="pt",
)

input_ids = encoded["input_ids"][0]

print("Tokens:", len(input_ids))

Tokens: 283


In [23]:
for token_id in input_ids:
    token = tokenizer.decode([token_id])
    
    if "assistant" in token or "<|im_end|>" in token:
        print(token_id, repr(token))

tensor(17847) ' assistant'
tensor(151645) '<|im_end|>'
tensor(151645) '<|im_end|>'
tensor(77091) 'assistant'
tensor(151645) '<|im_end|>'


In [24]:
print("Gold answer:")
print(format_answer(sft_train[0]["answers"]))

Gold answer:


NameError: name 'format_answer' is not defined